# STEP 2 — LEXICAL FEATURE EXTRACTION (58 features)
# Turkish Quishing Detection v9 — Full URL + 4-class keyword balance
# ============================================================
# Input  : dataset_with_folds_v9.csv  (1,239,308 rows, full URLs)
# Output : lexical_features_v9.csv     — 58 cols + metadata
#          lexical_feature_names.pkl   — ordered feature list
#          lexical_mi_scores.png       — top-20 MI ranking
#
# 58 features (was 54):
#   Added 4 class-balanced keyword features:
#     has_malware_keyword, num_malware_keywords
#     has_scam_keyword,    num_scam_keywords
#   These mirror the existing 4 phishing keyword features
#   Critical for 5-class experiment (malware/scam minority classes)
# ============================================================


In [ ]:
import os, re, math, pickle, warnings
from collections import Counter
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import tldextract
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif

# Import the keyword lists from classification_keywords.py
# Works in both .py scripts and Jupyter/Colab notebooks
import sys
KEYWORDS_DIR = "/content/drive/MyDrive/TUMC_dataset/lexicons"
if KEYWORDS_DIR not in sys.path:
    sys.path.insert(0, KEYWORDS_DIR)
from classification_keywords import (
    PHISHING_KEYWORDS as CK_PHISHING_KW,
    MALWARE_KEYWORDS  as CK_MALWARE_KW,
    SCAM_KEYWORDS     as CK_SCAM_KW,
)

# Turkish phishing terms now come from the curated, cleaned lexicon
# (single source of truth), NOT a hardcoded compact list. This keeps the
# has_turkish_keyword feature consistent with the lexicons validated in
# Step 0C and avoids the ad-hoc 26-word subset that could drift / collide.
from turkish_lexicons import TR_PHISHING as _TR_PHISHING_RAW
try:
    from lexicon_cleaner import clean_lexicon
    TR_PHISHING_LEX = clean_lexicon(_TR_PHISHING_RAW)
except Exception:
    TR_PHISHING_LEX = _TR_PHISHING_RAW

warnings.filterwarnings("ignore")

# ── PATHS ─────────────────────────────────────────────────────
DATA_DIR = (
    "/content/drive/MyDrive/TUMC_dataset/lexicons"
)
INPUT_FILE  = os.path.join(DATA_DIR, "dataset_with_folds_v10.csv")
OUTPUT_CSV  = os.path.join(DATA_DIR, "lexical_features_final.csv")
OUTPUT_PKL  = os.path.join(DATA_DIR, "lexical_feature_names_final.pkl")
PLOTS_DIR   = os.path.join(DATA_DIR, "eda_plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════
# KEYWORD CONSTANTS — built from classification_keywords.py
# Compiled once into regex patterns for fast vectorised matching
# ════════════════════════════════════════════════════════════

# Generic English/global phishing keywords (distinct signal from the
# Turkish lexicon: login/brand terms). Compiled token-aware via _compile.
PHISHING_KW_COMPACT = [
    "login","signin","verify","secure","account","update",
    "confirm","banking","paypal","password","credential",
    "webscr","submit","checkout","redirect",
]

# Pre-compile full keyword patterns from classification_keywords.py
KEEP_SUBSTRING_SHORT = {"iade", "onay", "iban", "2fa", "worm", "mhrs", "mlm", "odul"}
_KW_LETTER = r"[a-z\u00e7\u011f\u0131\u00f6\u015f\u00fc]"  # a-z + cgiosu
def _compile(keywords, short_len=5):
    # ALLOWLIST token-boundary matching (consistent with classification_keywords.py).
    # Genuine Turkish morphemes in KEEP_SUBSTRING_SHORT keep substring matching so
    # agglutination is preserved ("iade" in "vergiadesi"); other short tokens get
    # strict letter-boundaries so "gib" does not match inside "gibi", etc.
    parts = []
    for kw in keywords:
        if any(c in kw for c in r"\.^$*+?{}[]|()/"):
            parts.append(kw)
        elif " " in kw or "-" in kw:
            parts.append(re.escape(kw))
        elif len(kw) < short_len and kw not in KEEP_SUBSTRING_SHORT:
            parts.append(r"(?<!" + _KW_LETTER + r")" + re.escape(kw) + r"(?!" + _KW_LETTER + r")")
        else:
            parts.append(re.escape(kw))
    return re.compile("|".join(parts), re.IGNORECASE)

PATTERN_PHISHING_FULL = _compile(CK_PHISHING_KW)
PATTERN_MALWARE_FULL  = _compile(CK_MALWARE_KW)
PATTERN_SCAM_FULL     = _compile(CK_SCAM_KW)
PATTERN_TURKISH_FULL  = _compile(TR_PHISHING_LEX)   # curated Turkish lexicon
PATTERN_PHISHING_EN   = _compile(PHISHING_KW_COMPACT)  # English/global terms

SUSPICIOUS_TLDS = {
    "xyz","top","club","online","site","website","space",
    "info","click","link","live","stream","work","pw",
    "tk","ml","ga","cf","lat","cfd","sbs",
}

FREE_HOSTING = {
    "github.io","netlify.app","vercel.app","glitch.me",
    "000webhostapp.com","weebly.com","wixsite.com",
    "blogspot.com","wordpress.com","pages.dev","workers.dev",
    "duckdns.org","hopto.org","zapto.org","no-ip.com",
}

# ════════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════════
def url_entropy(s):
    s = str(s).strip()
    if not s:
        return 0.0
    freq = Counter(s)
    n = len(s)
    return -sum((c/n) * math.log2(c/n) for c in freq.values())

def safe_parse(url):
    try:
        u = str(url).strip()
        if not u.startswith(("http://","https://")):
            u = "http://" + u
        p = urlparse(u)
        _ = p.port
        return p
    except Exception:
        return None

# ════════════════════════════════════════════════════════════
# CANONICAL 58-FEATURE LIST (was 54 + 4 new)
# ════════════════════════════════════════════════════════════
LEXICAL_FEATURE_NAMES = [
    # Group 1: Original (9)
    "url_len","num_dots","num_slashes","num_digits",
    "num_specials","subdomain_len","domain_len","tld_len",
    "is_tr_domain",
    # Group 2: Ratios (3)
    "digit_ratio","alpha_ratio","special_ratio",
    # Group 3: Subdomain (3)
    "num_subdomains","has_www","subdomain_digits",
    # Group 4: Path (5)
    "path_len","path_depth","path_digits","has_php","has_exe",
    # Group 5: Query (3)
    "has_query","query_len","num_params",
    # Group 6: Chars (9)
    "num_hyphens","num_underscores","num_at_signs",
    "num_ampersands","num_equals","num_percent",
    "has_at_in_url","has_port","has_ip",
    # Group 7: Protocol (2)
    "is_https","has_double_slash",
    # Group 8: Entropy (3)
    "url_entropy","domain_entropy","path_entropy",
    # Group 9: Obfuscation (4)
    "has_repeated_chars","has_hex_encoding",
    "num_encoded_chars","has_double_dot",
    # Group 10: TR phishing keywords (2)
    "has_turkish_keyword","num_turkish_keywords",
    # Group 11: EN phishing keywords (2)
    "has_phishing_keyword","num_phishing_keywords",
    # Group 12: NEW — Malware keywords (2)  ← added
    "has_malware_keyword","num_malware_keywords",
    # Group 13: NEW — Scam keywords (2)     ← added
    "has_scam_keyword","num_scam_keywords",
    # Group 14: Domain signals (5)
    "domain_hyphen_count","domain_digit_count",
    "domain_has_number","is_suspicious_tld","is_free_hosting",
    # Group 15: Length bucket (1)
    "url_len_bucket",
    # Group 16: Tokens (3)
    "num_tokens","max_token_len","mean_token_len",
]
assert len(LEXICAL_FEATURE_NAMES) == 58, f"got {len(LEXICAL_FEATURE_NAMES)}"

# ════════════════════════════════════════════════════════════
# EXTRACT 58 FEATURES FROM ONE URL
# ════════════════════════════════════════════════════════════
def extract_lexical(url):
    u       = str(url).strip()
    ext     = tldextract.extract(u if u.startswith("http") else "http://" + u)
    parsed  = safe_parse(u)
    u_lower = u.lower()

    subdomain = ext.subdomain or ""
    domain    = ext.domain    or ""
    suffix    = ext.suffix    or ""
    path      = parsed.path   if parsed else ""
    query     = parsed.query  if parsed else ""
    netloc    = parsed.netloc if parsed else ""

    f = {}
    n = max(len(u), 1)

    # Group 1: Original (9)
    f["url_len"]       = len(u)
    f["num_dots"]      = u.count(".")
    f["num_slashes"]   = u.count("/")
    f["num_digits"]    = sum(c.isdigit() for c in u)
    f["num_specials"]  = sum(
        not c.isalnum() and c not in "://.-_" for c in u)
    f["subdomain_len"] = len(subdomain)
    f["domain_len"]    = len(domain)
    f["tld_len"]       = len(suffix)
    f["is_tr_domain"]  = int("tr" in suffix.lower().split("."))

    # Group 2: Ratios (3)
    f["digit_ratio"]   = sum(c.isdigit() for c in u) / n
    f["alpha_ratio"]   = sum(c.isalpha() for c in u) / n
    f["special_ratio"] = f["num_specials"] / n

    # Group 3: Subdomain (3)
    parts = subdomain.split(".") if subdomain else []
    f["num_subdomains"]   = len(parts) if subdomain else 0
    f["has_www"]          = int(
        subdomain.lower() == "www" or
        subdomain.lower().startswith("www."))
    f["subdomain_digits"] = sum(c.isdigit() for c in subdomain)

    # Group 4: Path (5)
    path_parts = [p for p in path.split("/") if p]
    f["path_len"]    = len(path)
    f["path_depth"]  = len(path_parts)
    f["path_digits"] = sum(c.isdigit() for c in path)
    f["has_php"]     = int(".php" in path.lower())
    f["has_exe"]     = int(
        ".exe" in path.lower() or
        ".zip" in path.lower() or
        ".bat" in path.lower() or
        ".msi" in path.lower())

    # Group 5: Query (3)
    f["has_query"]  = int(len(query) > 0)
    f["query_len"]  = len(query)
    f["num_params"] = len(query.split("&")) if query else 0

    # Group 6: Chars (9)
    f["num_hyphens"]     = u.count("-")
    f["num_underscores"] = u.count("_")
    f["num_at_signs"]    = u.count("@")
    f["num_ampersands"]  = u.count("&")
    f["num_equals"]      = u.count("=")
    f["num_percent"]     = u.count("%")
    f["has_at_in_url"]   = int("@" in u)
    try:
        _port = parsed.port if parsed else None
    except Exception:
        _port = None
    f["has_port"] = int(bool(_port))
    f["has_ip"]   = int(bool(re.match(
        r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$",
        netloc.split(":")[0])))

    # Group 7: Protocol (2)
    f["is_https"]         = int(u_lower.startswith("https"))
    f["has_double_slash"] = int("//" in path)

    # Group 8: Entropy (3)
    f["url_entropy"]    = round(url_entropy(u),      4)
    f["domain_entropy"] = round(url_entropy(domain), 4)
    f["path_entropy"]   = round(url_entropy(path),   4)

    # Group 9: Obfuscation (4)
    f["has_repeated_chars"] = int(bool(re.search(r"(.)\1{3,}", u)))
    f["has_hex_encoding"]   = int("%2" in u or "%3" in u)
    f["num_encoded_chars"]  = len(re.findall(r"%[0-9a-fA-F]{2}", u))
    f["has_double_dot"]     = int(".." in u)

    # Group 10: TR phishing keywords (2)
    _tr_hits = PATTERN_TURKISH_FULL.findall(u_lower)
    f["has_turkish_keyword"]  = int(len(_tr_hits) > 0)
    f["num_turkish_keywords"] = len(_tr_hits)

    # Group 11: EN phishing keywords (2)
    _ph_hits = PATTERN_PHISHING_EN.findall(u_lower)
    f["has_phishing_keyword"]  = int(len(_ph_hits) > 0)
    f["num_phishing_keywords"] = len(_ph_hits)

    # ── NEW Group 12: Malware keywords (2) ────────────────
    # Uses full PATTERN_MALWARE_FULL from classification_keywords
    # Counts unique regex matches in the URL
    mal_matches = PATTERN_MALWARE_FULL.findall(u_lower)
    f["has_malware_keyword"]  = int(len(mal_matches) > 0)
    f["num_malware_keywords"] = len(mal_matches)

    # ── NEW Group 13: Scam keywords (2) ───────────────────
    scam_matches = PATTERN_SCAM_FULL.findall(u_lower)
    f["has_scam_keyword"]  = int(len(scam_matches) > 0)
    f["num_scam_keywords"] = len(scam_matches)

    # Group 14: Domain signals (5)
    f["domain_hyphen_count"] = domain.count("-")
    f["domain_digit_count"]  = sum(c.isdigit() for c in domain)
    f["domain_has_number"]   = int(any(c.isdigit() for c in domain))
    f["is_suspicious_tld"]   = int(suffix.lower() in SUSPICIOUS_TLDS)
    full_host = f"{domain}.{suffix}".lower() if domain and suffix else ""
    f["is_free_hosting"] = int(any(fh in full_host for fh in FREE_HOSTING))

    # Group 15: URL length bucket (1)
    url_l = len(u)
    if   url_l <= 30:  f["url_len_bucket"] = 0
    elif url_l <= 60:  f["url_len_bucket"] = 1
    elif url_l <= 100: f["url_len_bucket"] = 2
    else:              f["url_len_bucket"] = 3

    # Group 16: Tokens (3)
    tokens = [t for t in re.split(r"[.\-/_?=&]", u) if t]
    f["num_tokens"]     = len(tokens)
    f["max_token_len"]  = max((len(t) for t in tokens), default=0)
    f["mean_token_len"] = round(
        sum(len(t) for t in tokens) / len(tokens), 4) if tokens else 0.0

    return {k: f[k] for k in LEXICAL_FEATURE_NAMES}


# ════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════
print("=" * 70)
print("STEP 2 — LEXICAL FEATURE EXTRACTION (58 features)")
print("=" * 70)

print(f"\n[1] Loading dataset ...")
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"    Rows: {len(df):,}")
print(f"    Cols: {list(df.columns)}")

if "url" not in df.columns:
    raise ValueError("Required column 'url' missing from input CSV")

# Sanity check
sample = df["url"].iloc[0]
sf = extract_lexical(sample)
print(f"\n[2] Sanity check on: '{sample[:60]}'")
print(f"    Features extracted: {len(sf)} ✓")

# Extract for all rows
print(f"\n[3] Extracting features for {len(df):,} URLs ...")
print(f"    (Expect 6-12 min on Colab — 1.24M URLs)")

records = []
for i, url in enumerate(df["url"]):
    records.append(extract_lexical(url))
    if (i + 1) % 50_000 == 0:
        pct = (i + 1) / len(df) * 100
        print(f"    {i+1:>9,} / {len(df):,}  ({pct:5.1f}%)")

feat_df = pd.DataFrame(records)
print(f"\n[4] Feature matrix: {feat_df.shape}")
print(f"    Nulls: {feat_df.isnull().sum().sum()}")

# Class-level statistics — both binary and 5-class
print(f"\n[5] Feature means by binary label:")
feat_df["label"]       = df["label"].values
feat_df["label_enc"]   = df["label_enc"].values
feat_df["class_final"] = df["class_final"].values
feat_df["class_enc"]   = df["class_enc"].values

key = ["url_len","url_entropy","digit_ratio","is_tr_domain","has_ip",
       "has_turkish_keyword","has_phishing_keyword",
       "has_malware_keyword","has_scam_keyword","is_https"]
print(feat_df.groupby("label")[key].mean().round(3).to_string())

print(f"\n[6] Feature means by 5-class:")
print(feat_df.groupby("class_final")[key].mean().round(3).to_string())

# Leakage warning
print(f"\n{'='*70}")
print("⚠  LEAKAGE RISK: is_tr_domain")
print(f"{'='*70}")
tr_b = feat_df[feat_df["label"]=="benign"]["is_tr_domain"].mean()
tr_m = feat_df[feat_df["label"]=="malicious"]["is_tr_domain"].mean()
print(f"   Benign  .tr rate: {tr_b*100:.1f}%")
print(f"   Malicious .tr   : {tr_m*100:.1f}%")
print(f"   Gap             : {(tr_b-tr_m)*100:.1f}pp")
print(f"   → Run Step 3 Exp A (with) and Exp B (without is_tr_domain)")

# Verify malware/scam features fire on correct classes
print(f"\n[7] Verifying new keyword features by class:")
print(f"    has_malware_keyword rates:")
for cls, sub in feat_df.groupby("class_final"):
    rate = sub["has_malware_keyword"].mean()
    print(f"      {cls:<20s}: {rate:.1%}")
print(f"    has_scam_keyword rates:")
for cls, sub in feat_df.groupby("class_final"):
    rate = sub["has_scam_keyword"].mean()
    print(f"      {cls:<20s}: {rate:.1%}")

# Save outputs
print(f"\n[8] Saving outputs ...")
for c in ["url","source","tld","fold","reg_domain"]:
    if c in df.columns:
        feat_df[c] = df[c].values

final_cols = LEXICAL_FEATURE_NAMES + [
    "url","source","tld","label","label_enc",
    "class_final","class_enc","fold","reg_domain"]
final_cols = [c for c in final_cols if c in feat_df.columns]
feat_df[final_cols].to_csv(OUTPUT_CSV, index=False)
print(f"    {OUTPUT_CSV}")

with open(OUTPUT_PKL, "wb") as fh:
    pickle.dump(LEXICAL_FEATURE_NAMES, fh)
print(f"    {OUTPUT_PKL}")

# Mutual information — binary
print(f"\n[9] Mutual information ranking (binary) ...")
X = feat_df[LEXICAL_FEATURE_NAMES].values
y_bin = df["label_enc"].values
mi_bin = mutual_info_classif(X, y_bin, random_state=42)
mi_bin_s = pd.Series(mi_bin, index=LEXICAL_FEATURE_NAMES).sort_values(ascending=False)
print(f"\n    Top 15 features by MI (binary):")
print(mi_bin_s.head(15).round(4).to_string())

# Mutual information — 5-class
print(f"\n[10] Mutual information ranking (5-class) ...")
y_5c = df["class_enc"].values
mi_5c = mutual_info_classif(X, y_5c, random_state=42)
mi_5c_s = pd.Series(mi_5c, index=LEXICAL_FEATURE_NAMES).sort_values(ascending=False)
print(f"\n    Top 15 features by MI (5-class):")
print(mi_5c_s.head(15).round(4).to_string())

# Show where the 4 new features rank
print(f"\n[11] New keyword features ranking:")
new_features = ["has_malware_keyword","num_malware_keywords",
                "has_scam_keyword","num_scam_keywords"]
for f_name in new_features:
    bin_rank = list(mi_bin_s.index).index(f_name) + 1
    fc_rank  = list(mi_5c_s.index).index(f_name) + 1
    print(f"    {f_name:<25s}: binary rank #{bin_rank}, 5-class rank #{fc_rank}")

# Comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 9))
mi_bin_s.head(20).plot(kind="barh", ax=ax1, color="#1D9E75", edgecolor="none")
ax1.invert_yaxis()
ax1.set_xlabel("MI score (binary)")
ax1.set_title("Top 20 — Binary classification")
mi_5c_s.head(20).plot(kind="barh", ax=ax2, color="#5B7BB8", edgecolor="none")
ax2.invert_yaxis()
ax2.set_xlabel("MI score (5-class)")
ax2.set_title("Top 20 — 5-class classification")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "lexical_mi_scores.png"), dpi=150)
plt.close()
print(f"    Saved: {PLOTS_DIR}/lexical_mi_scores.png")

print(f"""
{'='*70}
STEP 2 COMPLETE — 58 features
{'='*70}
  Feature groups:
    Original 54 features (length, ratios, entropy, etc.)
    +2 malware keyword features (NEW)
    +2 scam keyword features    (NEW)

  Top discriminator (binary): {mi_bin_s.index[0]}
  Top discriminator (5class): {mi_5c_s.index[0]}

  Outputs:
    {OUTPUT_CSV}
    {OUTPUT_PKL}
    {PLOTS_DIR}/lexical_mi_scores.png

  Next → Step 2B: adversarial features (brand mimicry, typo detection)
{'='*70}
""")

STEP 2 — LEXICAL FEATURE EXTRACTION (58 features)

[1] Loading dataset ...
    Rows: 1,239,308
    Cols: ['url', 'source', 'tld', 'class_final', 'class_enc', 'label', 'label_enc', 'phishing_score', 'malware_score', 'scam_score', 'matched_keywords', 'reg_domain', 'fold']

[2] Sanity check on: 'http://afsaces.accesscam.org'
    Features extracted: 58 ✓

[3] Extracting features for 1,239,308 URLs ...
    (Expect 6-12 min on Colab — 1.24M URLs)
       50,000 / 1,239,308  (  4.0%)
      100,000 / 1,239,308  (  8.1%)
      150,000 / 1,239,308  ( 12.1%)
      200,000 / 1,239,308  ( 16.1%)
      250,000 / 1,239,308  ( 20.2%)
      300,000 / 1,239,308  ( 24.2%)
      350,000 / 1,239,308  ( 28.2%)
      400,000 / 1,239,308  ( 32.3%)
      450,000 / 1,239,308  ( 36.3%)
      500,000 / 1,239,308  ( 40.3%)
      550,000 / 1,239,308  ( 44.4%)
      600,000 / 1,239,308  ( 48.4%)
      650,000 / 1,239,308  ( 52.4%)
      700,000 / 1,239,308  ( 56.5%)
      750,000 / 1,239,308  ( 60.5%)
      800,000 /